# Python for Data Analysis - Training Notebook

This notebook is a hands-on companion for the PyTrack course. It moves from Python basics to NumPy, pandas, cleaning, analysis, visualization, and a small machine-learning-style workflow.

Run the cells from top to bottom. Each section includes a short demo plus a practice task.

## 1. Setup and Sample Data

The notebook uses a small built-in sales dataset so no file download is required.

In [ ]:
import math
import statistics
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

sales_records = [
    {"date": "2026-01-03", "city": "Cairo", "product": "Laptop", "units": 4, "price": 15500, "channel": "Online", "rating": 4.7},
    {"date": "2026-01-05", "city": "Alex", "product": "Headset", "units": 12, "price": 750, "channel": "Store", "rating": 4.1},
    {"date": "2026-01-08", "city": "Giza", "product": "Mouse", "units": 20, "price": 320, "channel": "Online", "rating": 4.3},
    {"date": "2026-01-12", "city": "Cairo", "product": "Keyboard", "units": 9, "price": 1100, "channel": "Store", "rating": 4.5},
    {"date": "2026-01-18", "city": "Mansoura", "product": "Monitor", "units": 5, "price": 6200, "channel": "Online", "rating": 4.6},
    {"date": "2026-01-22", "city": "Alex", "product": "Laptop", "units": 3, "price": 16200, "channel": "Online", "rating": 4.8},
    {"date": "2026-02-02", "city": "Cairo", "product": "Mouse", "units": 18, "price": 340, "channel": "Store", "rating": 4.0},
    {"date": "2026-02-09", "city": "Giza", "product": "Headset", "units": 11, "price": 790, "channel": "Online", "rating": 4.2},
    {"date": "2026-02-16", "city": "Mansoura", "product": "Keyboard", "units": 7, "price": 1150, "channel": "Store", "rating": 4.4},
    {"date": "2026-02-21", "city": "Cairo", "product": "Monitor", "units": 6, "price": 6500, "channel": "Online", "rating": 4.9},
]

df = pd.DataFrame(sales_records)
df

## 2. Python Fundamentals

Use variables, lists, dictionaries, loops, and functions to describe a small business rule.

In [ ]:
tax_rate = 0.14
discount_threshold = 50000
discount_rate = 0.05

def invoice_total(units, price):
    subtotal = units * price
    discount = subtotal * discount_rate if subtotal >= discount_threshold else 0
    tax = (subtotal - discount) * tax_rate
    return round(subtotal - discount + tax, 2)

for record in sales_records[:4]:
    total = invoice_total(record["units"], record["price"])
    print(f"{record['city']:8s} | {record['product']:8s} | invoice total: {total:,.2f}")

**Practice:** Change `tax_rate` or `discount_rate`, then rerun the cell. Which product is most affected by the change?

## 3. NumPy Arrays

NumPy is useful when you need fast numeric calculations.

In [ ]:
units = df["units"].to_numpy()
prices = df["price"].to_numpy()
revenue = units * prices

print("Revenue values:", revenue)
print("Total revenue:", revenue.sum())
print("Average revenue:", round(revenue.mean(), 2))
print("Highest revenue row index:", revenue.argmax())

above_average = revenue > revenue.mean()
print("Rows above average revenue:", above_average)

**Practice:** Create an array called `revenue_with_tax` that adds 14 percent tax to every revenue value.

In [ ]:
# TODO: complete this practice cell
revenue_with_tax = revenue * 1.14
revenue_with_tax

## 4. Pandas Basics

pandas helps you inspect, filter, sort, and summarize tabular data.

In [ ]:
df["date"] = pd.to_datetime(df["date"])
df["revenue"] = df["units"] * df["price"]
df["month"] = df["date"].dt.to_period("M").astype(str)

display(df.head())
display(df.info())
display(df.select_dtypes(include="number").describe())

In [ ]:
online_high_value = df[(df["channel"] == "Online") & (df["revenue"] >= 10000)]
online_high_value.sort_values("revenue", ascending=False)

**Practice:** Filter the rows where `city` is `Cairo` and show only `date`, `product`, `units`, and `revenue`.

In [ ]:
# TODO: complete this practice cell
df.loc[df["city"] == "Cairo", ["date", "product", "units", "revenue"]]

## 5. GroupBy, Pivot Tables, and Reporting

Grouped summaries turn raw rows into business answers.

In [ ]:
city_summary = (
    df.groupby("city", as_index=False)
      .agg(total_units=("units", "sum"), total_revenue=("revenue", "sum"), avg_rating=("rating", "mean"))
      .sort_values("total_revenue", ascending=False)
)

city_summary

In [ ]:
pivot = pd.pivot_table(
    df,
    values="revenue",
    index="city",
    columns="channel",
    aggfunc="sum",
    fill_value=0,
)

pivot

**Practice:** Build a product summary with total units, total revenue, and average price.

In [ ]:
# TODO: complete this practice cell
product_summary = (
    df.groupby("product", as_index=False)
      .agg(total_units=("units", "sum"), total_revenue=("revenue", "sum"), avg_price=("price", "mean"))
      .sort_values("total_revenue", ascending=False)
)

product_summary

## 6. Data Cleaning

Real datasets often contain messy text, missing values, duplicates, and incorrect types.

In [ ]:
messy = pd.DataFrame({
    "customer": [" Mona ", "ALI", "sara", "ALI", None],
    "city": ["cairo", "Alex", "giza", "Alex", "cairo"],
    "age": [28, 34, None, 34, 120],
    "signup_date": ["2026/01/05", "2026-01-07", "bad date", "2026-01-07", "2026-02-01"],
})

messy

In [ ]:
clean = messy.copy()
clean["customer"] = clean["customer"].str.strip().str.title()
clean["city"] = clean["city"].str.strip().str.title()
clean["age"] = clean["age"].where(clean["age"].between(0, 100))
clean["age"] = clean["age"].fillna(clean["age"].median())
clean["signup_date"] = pd.to_datetime(clean["signup_date"], errors="coerce")
clean = clean.drop_duplicates()

clean

**Practice:** Add a boolean column called `has_valid_signup_date` before dropping rows with invalid dates.

## 7. Visualization

Good charts make comparisons faster.

In [ ]:
plt.figure(figsize=(8, 4))
plt.bar(city_summary["city"], city_summary["total_revenue"])
plt.title("Revenue by City")
plt.xlabel("City")
plt.ylabel("Revenue")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
monthly_revenue = df.groupby("month", as_index=False)["revenue"].sum()

plt.figure(figsize=(8, 4))
plt.plot(monthly_revenue["month"], monthly_revenue["revenue"], marker="o")
plt.title("Monthly Revenue")
plt.xlabel("Month")
plt.ylabel("Revenue")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

**Practice:** Create a bar chart that compares total revenue by product.

## 8. Mini Machine Learning Workflow

This simple example uses linear regression ideas to predict revenue from units and price. It uses NumPy only, so it can run in most notebook environments.

In [ ]:
X = df[["units", "price"]].to_numpy(dtype=float)
y = df["revenue"].to_numpy(dtype=float)

# Add an intercept column for a basic least-squares model.
X_design = np.column_stack([np.ones(len(X)), X])
coefficients, residuals, rank, singular_values = np.linalg.lstsq(X_design, y, rcond=None)

predictions = X_design @ coefficients
mae = np.mean(np.abs(y - predictions))

print("Intercept:", round(coefficients[0], 2))
print("Units coefficient:", round(coefficients[1], 2))
print("Price coefficient:", round(coefficients[2], 2))
print("Mean absolute error:", round(mae, 2))

pd.DataFrame({"actual": y, "predicted": predictions.round(2), "error": (y - predictions).round(2)})

**Practice:** Add `rating` as a third feature. Does the mean absolute error improve?

## 9. Git, GitHub, and Reproducible Projects

Data analysts need more than code. A clean project should be easy to rerun, review, and share.

A simple project structure:

```text
project-name/
  data/
  notebooks/
  src/
  reports/
  README.md
  requirements.txt
```

Useful Git commands:

```bash
git status
git add notebooks/training_notebook.ipynb
git commit -m "Add analysis notebook"
git log --oneline
```

**Practice:** Write a short README section that explains what this sales analysis does and how to run it.

In [ ]:
project_readme = """
# Sales Analysis Project

This project analyzes sales by city, product, channel, and month.

Steps:
1. Load the data.
2. Clean date and revenue columns.
3. Summarize revenue by category.
4. Create charts and recommendations.
"""

print(project_readme)

## 10. Exploratory Data Analysis Checklist

Exploratory Data Analysis, usually called EDA, helps you understand a dataset before modeling or reporting.

Core questions:

1. How many rows and columns are there?
2. What are the column types?
3. Are there missing values or duplicates?
4. What are the main numeric distributions?
5. Which categories dominate the data?
6. Are there suspicious outliers?
7. What business question should the analysis answer?

In [ ]:
def eda_summary(data):
    summary = {
        "rows": data.shape[0],
        "columns": data.shape[1],
        "duplicate_rows": int(data.duplicated().sum()),
        "missing_values": data.isna().sum().to_dict(),
        "numeric_columns": data.select_dtypes(include="number").columns.tolist(),
        "category_columns": data.select_dtypes(include="object").columns.tolist(),
    }
    return summary

eda_summary(df)

In [ ]:
for column in ["city", "product", "channel"]:
    print(f"Top values for {column}:")
    print(df[column].value_counts())
    print()

**Practice:** Add one EDA check that finds the minimum and maximum date in the dataset.

In [ ]:
# TODO: complete this practice cell
print("First date:", df["date"].min())
print("Last date:", df["date"].max())

## 11. Statistics and Probability for Analysts

A data analyst should understand variation, averages, sampling, and uncertainty. These ideas help you avoid overreacting to noisy data.

In [ ]:
revenue_mean = df["revenue"].mean()
revenue_median = df["revenue"].median()
revenue_std = df["revenue"].std()

print("Mean revenue:", round(revenue_mean, 2))
print("Median revenue:", round(revenue_median, 2))
print("Standard deviation:", round(revenue_std, 2))

# A simple bootstrap confidence interval for average revenue.
rng = np.random.default_rng(42)
bootstrap_means = []
for _ in range(1000):
    sample = rng.choice(df["revenue"].to_numpy(), size=len(df), replace=True)
    bootstrap_means.append(sample.mean())

lower, upper = np.percentile(bootstrap_means, [2.5, 97.5])
print(f"Approx. 95% confidence interval: {lower:,.0f} to {upper:,.0f}")

**Practice:** Compare average revenue for Online vs Store orders. Which channel has the higher average?

In [ ]:
# TODO: complete this practice cell
df.groupby("channel")["revenue"].mean().sort_values(ascending=False)

## 12. Advanced SQL with Python

SQL is essential for analysts because many business datasets live in databases. You can practice SQL directly in Python with SQLite.

In [ ]:
import sqlite3

connection = sqlite3.connect(":memory:")
df.to_sql("sales", connection, index=False, if_exists="replace")

query = """
SELECT
    city,
    channel,
    SUM(revenue) AS total_revenue,
    SUM(units) AS total_units,
    ROUND(AVG(rating), 2) AS avg_rating
FROM sales
GROUP BY city, channel
ORDER BY total_revenue DESC;
"""

sql_summary = pd.read_sql_query(query, connection)
sql_summary

In [ ]:
window_query = """
SELECT
    product,
    city,
    revenue,
    RANK() OVER (PARTITION BY product ORDER BY revenue DESC) AS product_revenue_rank
FROM sales
ORDER BY product, product_revenue_rank;
"""

pd.read_sql_query(window_query, connection)

**Practice:** Write a SQL query that returns total revenue by product.

In [ ]:
# TODO: complete this practice cell
product_sql = """
SELECT product, SUM(revenue) AS total_revenue
FROM sales
GROUP BY product
ORDER BY total_revenue DESC;
"""

pd.read_sql_query(product_sql, connection)

## 13. APIs, JSON, and Web Data

Many analytics workflows start by receiving JSON from an API. JSON usually looks like Python dictionaries and lists.

In [ ]:
api_response = {
    "status": "ok",
    "source": "sample sales api",
    "data": [
        {"city": "Cairo", "orders": 120, "satisfaction": 4.6},
        {"city": "Alex", "orders": 95, "satisfaction": 4.2},
        {"city": "Giza", "orders": 80, "satisfaction": 4.3},
    ],
}

api_df = pd.DataFrame(api_response["data"])
api_df

In [ ]:
merged = city_summary.merge(api_df, on="city", how="left")
merged

**Practice:** Add a calculated column called `revenue_per_order` to `merged`.

In [ ]:
# TODO: complete this practice cell
merged["revenue_per_order"] = merged["total_revenue"] / merged["orders"]
merged

## 14. Dashboards and Portfolio Projects

After analysis, analysts need to communicate results. A simple dashboard or portfolio project should include:

1. A clear question.
2. A cleaned dataset.
3. KPIs such as revenue, orders, and rating.
4. 2-4 useful charts.
5. A short recommendation.
6. A README with setup instructions.

Example Streamlit starter:

```python
import streamlit as st
import pandas as pd

st.title("Sales Dashboard")
st.metric("Total Revenue", f"{df['revenue'].sum():,.0f}")
st.bar_chart(df.groupby('city')['revenue'].sum())
```

**Practice:** Design three KPIs for a final dashboard based on the sales dataset.

In [ ]:
dashboard_kpis = {
    "total_revenue": df["revenue"].sum(),
    "total_units": df["units"].sum(),
    "average_rating": round(df["rating"].mean(), 2),
}

dashboard_kpis

## 15. Final Challenge

Use the sales dataset to answer these questions:

1. Which city has the highest total revenue?
2. Which product has the highest average rating?
3. Which channel sells the most units?
4. Create one chart that supports one of your answers.
5. Write a short recommendation for the business.

In [ ]:
# Final challenge workspace
# Write your analysis below.
